# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [ ]:
%pip install gymnasium[classic-control] stable-baselines3 wandb tsilva-notebook-utils==0.0.104 --quiet

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [ ]:
import torch

# TODO: move to utils
def get_default_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    elif torch.cuda.is_available(): return torch.device("cuda")
    else: return torch.device("cpu")

DEVICE = get_default_device()
print(f"Device set to: {DEVICE}")

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        max_epochs=-1,          # Maximum number of training epochs (-1 for no limit)
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        episodes_per_epoch=20, # Number of episodes per training epoch
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=20,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        updates_per_epoch=10,      # Number of epochs to update policy per training step
        mean_reward_window=100,  # Window size for mean reward calculation (for early stopping)
        n_envs="auto"          # Maximum number of parallel environments
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount for CartPole
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for CartPole
            minibatch_size=64,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            # TODO: restore this
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for CartPole
            entropy_coef=0.01     # Typical entropy for CartPole
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            episodes_per_epoch=24,  # More episodes per epoch for better sampling
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=128,         # Larger network for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }
    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    return {**common, **env_specific[env_id]}

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = setup_config(ENV_ID)
CONFIG

In [ ]:
# Set random seed for reproducibility
set_random_seed(CONFIG['seed'])

# Wrap build env with config parameters
build_env = lambda seed: _build_env(
    CONFIG['env_id'], 
    norm_obs=CONFIG['normalize'], 
    n_envs=CONFIG['n_envs'], seed=seed
)

# Test building env
env = build_env(CONFIG['seed'])
# TODO: encapsulate this
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.distributions import Categorical

# Local helper ------------------------------------------------------
def _get_device(module: torch.nn.Module) -> torch.device:  # type: ignore
    """Return the device of *module*'s first parameter."""
    return next(module.parameters()).device

def collect_rollouts(
    env,
    policy_model: torch.nn.Module,
    value_model: Optional[torch.nn.Module] = None,
    n_episodes: int = 1,
    deterministic: bool = False,
    collect_frames: bool = False,
    gamma: float = 0.99,
    gae_lambda: float = 0.95,
    normalize_advantage: bool = True,
    adv_norm_eps: float = 1e-8
):
    # Assert valid parameters
    if n_episodes < 1: raise ValueError("n_episodes must be ≥ 1")

    device = _get_device(policy_model)  # Get device from policy model

    # Create the episode buffers
    n_envs = env.num_envs
    ep_states      = [[] for _ in range(n_envs)]
    ep_actions     = [[] for _ in range(n_envs)]
    ep_rewards     = [[] for _ in range(n_envs)]
    ep_dones       = [[] for _ in range(n_envs)]
    ep_logps       = [[] for _ in range(n_envs)]
    ep_values      = [[] for _ in range(n_envs)]
    ep_advantages  = [[] for _ in range(n_envs)]
    ep_returns     = [[] for _ in range(n_envs)]
    ep_frames      = [[] for _ in range(n_envs)]

    # Global rollout buffers -----------------------------------------
    states = []
    actions = []
    rewards = []
    dones = []
    logps = []
    values = []
    advantages = []
    returns = []
    frames = []

    state = env.reset()
    episodes_collected = 0
    while episodes_collected < n_episodes:
        # Policy forward: get action predictions for current state
        state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
        with torch.no_grad(): policy_logits = policy_model(state_t)

        # Sample action from policy distribution 
        # (argmax instead of sampling if deterministic)
        policy_dist = Categorical(logits=policy_logits)
        action_t = policy_logits.argmax(dim=-1) if deterministic else policy_dist.sample()
        logp_t = policy_dist.log_prob(action_t)
        action_np = action_t.cpu().numpy()
        logp_np = logp_t.cpu().numpy()
        
        # Value forward: get value estimate for current state
        with torch.no_grad(): value_np = value_model(state_t).squeeze(-1).cpu().numpy() if value_model is not None else np.zeros(n_envs, dtype=np.float32)

        # Perform selected action on environment
        next_state, reward, done, _ = env.step(action_np)

        # Collect frames if requested (eg: for rendering episode)
        frame = env.get_images() if collect_frames else [0 for _ in range(n_envs)]

        # Per-env bookkeeping -----------------------------------
        for env_idx in range(n_envs):
            # Pick buffers for current environment index
            _ep_states      = ep_states[env_idx]
            _ep_actions     = ep_actions[env_idx]
            _ep_rewards     = ep_rewards[env_idx]
            _ep_dones       = ep_dones[env_idx]
            _ep_logps       = ep_logps[env_idx]
            _ep_values      = ep_values[env_idx]
            _ep_advantages  = ep_advantages[env_idx]
            _ep_returns     = ep_returns[env_idx]
            _ep_frames      = ep_frames[env_idx]
            
            # Pick step data for current environment index
            _env_state      = state[env_idx]
            _env_action     = action_np[env_idx]
            _env_reward     = reward[env_idx]
            _env_done       = done[env_idx]
            _env_logp       = logp_np[env_idx]
            _env_value      = value_np[env_idx]
            _env_frame      = frame[env_idx]

            # TODO: is there a need for these? can't we just add directly to global ones?
            # Store step data in episode buffers
            _ep_states.append(_env_state)
            _ep_actions.append(int(_env_action))
            _ep_rewards.append(float(_env_reward))
            _ep_dones.append(bool(_env_done))
            _ep_logps.append(float(_env_logp))
            _ep_values.append(float(_env_value))
            _ep_advantages.append(0.0)
            _ep_returns.append(0.0)
            _ep_frames.append(_env_frame)

            # Episode still running -----------------------------
            if not _env_done:
                continue

            # -------- GAE advantages ---------------------------
            next_gae   = 0.0
            next_value = 0.0
            for t in reversed(range(len(_ep_rewards))):
                mask       = 1.0 - _ep_dones[t]          # <-- add this
                delta      = _ep_rewards[t] + gamma * next_value * mask - _ep_values[t]
                _ep_advantages[t] = delta + gamma * gae_lambda * next_gae * mask
                next_gae   = _ep_advantages[t]
                next_value = _ep_values[t]

            # -------- λ-returns via (A + V) --------------------
            for t in range(len(_ep_returns)):
                _ep_returns[t] = _ep_advantages[t] + _ep_values[t]

            # Flush to global buffers ---------------------------
            states     .extend(_ep_states);     _ep_states.clear()
            actions    .extend(_ep_actions);    _ep_actions.clear()
            rewards    .extend(_ep_rewards);    _ep_rewards.clear()
            dones      .extend(_ep_dones);      _ep_dones.clear()
            logps      .extend(_ep_logps);      _ep_logps.clear()
            values     .extend(_ep_values);     _ep_values.clear()
            advantages .extend(_ep_advantages); _ep_advantages.clear()
            returns   .extend(_ep_returns);    _ep_returns.clear()
            frames.extend(list(_ep_frames)); _ep_frames.clear() 

            episodes_collected += 1

        state = next_state  # next vector-step

    # TODO: normalize advantages during training
    # -------- optional advantage normalisation (across batch) -------
    if normalize_advantage and advantages:
        adv_np = np.asarray(advantages, dtype=np.float32)
        adv_np = (adv_np - adv_np.mean()) / (adv_np.std() + adv_norm_eps)
        advantages[:] = adv_np.tolist()

    trajectories = (states, actions, rewards, dones, logps, values, advantages, returns, frames)

    # TODO: consider just returning trajectories and creating util that groups values back into episodes
    # ----------------------------------------------------------------
    return trajectories

policy_model = PolicyNet(obs_dim, act_dim, hidden_dim=CONFIG['hidden_dim']).to(DEVICE)
value_model = ValueNet(obs_dim, hidden_dim=CONFIG['hidden_dim']).to(DEVICE)
trajectories = collect_rollouts( # trajectories = (states, actions, rewards, dones, logps, values, adv, returns)
    env,
    policy_model,
    value_model,
    n_episodes=16,
    deterministic=False,
    collect_frames=False
)
len(trajectories)

In [ ]:
def group_trajectories_by_episode(trajectories):
    episodes = []
    steps = list(zip(*trajectories))
    episode = []
    for step in steps:
        episode.append(step)
        done = step[3]
        if not done: continue
        episodes.append(episode)
        episode = []
    return episodes

episodes = group_trajectories_by_episode(trajectories)
total_episode_rewards = [sum(ep[2] for ep in episode) for episode in episodes]
total_episode_rewards

In [ ]:
from torch.utils.data import Dataset

# TODO: should dataloader move to gpu?
class RolloutDataset(Dataset):
    """Holds PPO roll-out tensors and lets them be swapped in-place."""
    def __init__(self):
        self.trajectories = None 

    def update(self, *trajectories):
        self.trajectories = trajectories

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        return tuple(t[idx] for t in self.trajectories)

In [ ]:
import torch
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from torch.distributions import Categorical
from collections import deque

# ---------------------------------------------------------------------
#  PPO Lightning module
#  (assumes PolicyNet, ValueNet, build_env, collect_rollouts are defined)
# ---------------------------------------------------------------------
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()
        self.save_hyperparameters()

        # ----------------- unpack config -----------------
        self.config = config
        self.entropy_coef   = config['entropy_coef']
        self.clip_epsilon   = config['clip_epsilon']
        self.gamma          = config['gamma']
        self.lam            = config['lam']
        self.episodes_per_epoch = config['episodes_per_epoch']
        self.minibatch_size     = config['minibatch_size']
        self.eval_interval      = config['eval_interval']
        self.eval_episodes      = config['eval_episodes']
        self.reward_threshold   = config['reward_threshold']
        self.updates_per_epoch  = config['updates_per_epoch']
        self.policy_lr          = config['policy_lr']
        self.value_lr           = config['value_lr']
        self.mean_reward_window = config['mean_reward_window']

        # ----------------- models & env -----------------
        self.policy_model = PolicyNet(
            obs_dim, act_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.value_model = ValueNet(
            obs_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.env = build_env(config['seed']) # TODO: read from params
        self.obs_dim, self.act_dim = obs_dim, act_dim

        # ----------------- rollout storage --------------
        self.rollout_ds = RolloutDataset()   # <-- created once

        # TODO: softcode
        self.episode_reward_deque = deque(maxlen=self.mean_reward_window)  # Store recent mean rewards for early stopping
        
        # we’ll step the two optimizers manually
        self.automatic_optimization = False

    # ===================================================
    #  Lightning hooks
    # ===================================================
    def setup(self, stage: str):
        """Collect an initial roll-out before dataloaders are requested."""
        if stage == "fit": self._collect_and_store_rollout()   # fills rollout_ds

    # TODO: is this called only once?
    def train_dataloader(self):
        """Standard DataLoader built *once*; dataset is mutable."""
        return DataLoader(
            self.rollout_ds,
            batch_size=self.minibatch_size,
            shuffle=True
        )

    def on_train_epoch_start(self):
        """Refresh roll-out tensors in-place each epoch."""
        self._collect_and_store_rollout()

    # ---------------------------------------------------
    def training_step(self, batch, batch_idx):
        opt_policy, opt_value = self.optimizers()

        # unpack batch
        (states, actions, rewards, dones, old_logps, values, advantages, returns_, frames) = batch

        # main PPO loop
        policy_losses, value_losses = [], []
        for _ in range(self.updates_per_epoch):
            logits = self.policy_model(states)
            dist   = Categorical(logits=logits)
            new_logps = dist.log_prob(actions)

            ratio = torch.exp(new_logps - old_logps)
            surr1 = ratio * advantages
            surr2 = torch.clamp(
                ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon
            ) * advantages
            entropy = dist.entropy().mean() # TODO: log policy_entropy
            policy_loss = -(torch.min(surr1, surr2).mean() + self.entropy_coef * entropy)
            
            # TODO: try just maximizing the advantages (must remove previous calculations)
            #policy_loss = -advantages.mean()

            # ---------- value loss -----------
            value_pred  = self.value_model(states).squeeze()
            value_loss  = ((returns_ - value_pred) ** 2).mean()

            # ---------- optimizers ----------
            opt_policy.zero_grad()
            self.manual_backward(policy_loss)
            opt_policy.step()

            opt_value.zero_grad()
            self.manual_backward(value_loss)
            opt_value.step()

            policy_losses.append(policy_loss.detach())
            value_losses.append(value_loss.detach())

        mean_policy_loss = torch.stack(policy_losses).mean()
        mean_value_loss  = torch.stack(value_losses).mean()

        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) >= self.mean_reward_window else None

        # ---------- logging / early stop --------------
        log_dict = {
            'train/policy_loss': mean_policy_loss,
            'train/value_loss' : mean_value_loss,
        }
        if mean_reward is not None: log_dict['train/mean_reward'] = mean_reward
        self.log_dict(log_dict, prog_bar=True)

        if mean_reward is not None and mean_reward >= self.reward_threshold:
            print(f"Early stopping at epoch {self.current_epoch} with mean reward {mean_reward:.2f} >= threshold {self.reward_threshold}")
            self.trainer.should_stop = True

        return mean_policy_loss + mean_value_loss

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.value_lr)
        ]

    # ===================================================
    #  helpers
    # ===================================================
    def _collect_and_store_rollout(self):
        """Run episodes and refresh stored tensors on the model’s device."""
        
        # Collect rollouts and add them to 
        # # dataset for sampling during training step
        trajectories = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model,
            n_episodes      = self.episodes_per_epoch,
            deterministic   = False,
            collect_frames  = False
        )
        self.rollout_ds.update(*trajectories)

        # Add mean episode rewards to deque 
        # (to be able to average over N episodes)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards: self.episode_reward_deque.append(r)
    
    # ---------------------------------------------------
    def forward(self, x):
        return self.policy_model(x)

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=1,
    enable_progress_bar=True,
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode_frames
import random

# TODO: encapsulate this
# TODO: bug received episodes is bigger than requested n_episodes
trajectories = collect_rollouts(
    build_env(random.randint(0, 1_000_000)),
    ppo_agent.policy_model,
    n_episodes=4,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories)
print(len(episodes))
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(len(episode_frames))
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))